# HotpotQA GKN Benchmark

This notebook builds a public-benchmark retrieval comparison using a HotpotQA sample and an embedding-based vector store.

It compares:
- embedding baseline retrieval
- embedding retrieval + GKN hybrid retrieval

The notebook is intentionally focused on **retrieval evaluation**, not full answer generation.

## 1. Setup and configuration

This step loads the repo configuration, reads your local `.env` settings, and prepares artifact output folders.

The notebook now supports multiple embedding modes:
- `small` = Azure/OpenAI-compatible `text-embedding-3-small`
- `large` = Azure/OpenAI-compatible `text-embedding-3-large`
- `local` = sentence-transformers local model

It also supports cached vector store reuse to reduce repeated embedding calls.

In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path.cwd().parent / 'src'))

from geometric_knowledge_network.config import GKNConfig
from geometric_knowledge_network.evaluation import evaluate_retrieval
from geometric_knowledge_network.extraction import ConceptExtractor
from geometric_knowledge_network.graph_builder import KnowledgeNetworkBuilder
from geometric_knowledge_network.hotpotqa_loader import HotpotQALoader
from geometric_knowledge_network.hybrid_retriever import HybridRetriever
from geometric_knowledge_network.reporting import ArtifactManager
from geometric_knowledge_network.vector_store import EmbeddingVectorStore

config = GKNConfig()
artifacts = ArtifactManager(config.artifacts_dir)
run_id = artifacts.timestamp()

def status(message: str):
    print(f'[DONE] {message}')

status(f'Configuration loaded. Run ID: {run_id}')
print('HotpotQA path:', config.hotpotqa_data_path)
print('Path exists:', Path(config.hotpotqa_data_path).exists())
print('Sample size:', config.hotpotqa_sample_size)
print('Random seed:', config.hotpotqa_random_seed)
print('Embedding choice:', config.embedding_choice)
print('Embedding model:', config.openai_embedding_model if config.embedding_choice != 'local' else config.local_embedding_model)
print('Force rebuild vector store:', config.force_rebuild_vector_store)
print('Checkpoints dir:', config.checkpoints_dir)
print('Base URL set:', bool(config.openai_base_url))
print('API key set:', bool(config.openai_api_key))

[DONE] Configuration loaded. Run ID: 20260614_161438
HotpotQA path: C:\Users\minwuu01\GKN\data\hotpot_train_v1.1.json
Path exists: True
Sample size: 999
Random seed: 42
Embedding choice: small
Embedding model: text-embedding-3-small
Force rebuild vector store: False
Checkpoints dir: C:\Users\minwuu01\GKN\checkpoints
Base URL set: True
API key set: True


## 2. Load HotpotQA sample and construct a retrieval corpus

This step converts a sampled subset of HotpotQA into a graph-compatible retrieval corpus.

Sampling is now reproducible through a fixed random seed.

In [2]:
loader = HotpotQALoader()
documents, chunks, samples = loader.load_samples(
    config.hotpotqa_data_path,
    sample_size=config.hotpotqa_sample_size,
    random_seed=config.hotpotqa_random_seed,
)
status(f'Loaded {len(samples)} HotpotQA samples, {len(documents)} documents, and {len(chunks)} chunks.')

for sample in samples[:3]:
    print('-' * 80)
    print('Question ID:', sample.question_id)
    print('Question:', sample.question)
    print('Supporting titles:', sample.supporting_titles)

[DONE] Loaded 999 HotpotQA samples, 9750 documents, and 20754 chunks.
--------------------------------------------------------------------------------
Question ID: 5a7bb1315542997c3ec97253
Question: Which Emmett's Mark actor also played in the HBO series "The Wire"?
Supporting titles: ["Emmett's Mark", 'John Doman']
--------------------------------------------------------------------------------
Question ID: 5ae03a3655429942ec259c50
Question: Who directed the upcoming British action comedy film which has Johnny English as the first part? 
Supporting titles: ['Johnny English (film series)', 'Johnny English 3']
--------------------------------------------------------------------------------
Question ID: 5ab77e6655429928e1fe385d
Question: What conference has the most valuable NBA franchise according to Forbes?
Supporting titles: ['2008 NBA Playoffs', 'Los Angeles Lakers']


## 3. Build or load the embedding baseline and construct the GKN graph

This step builds the embedding-based vector store, or loads a cached checkpoint if available and rebuild is not forced.

If you are using a rate-limited cloud embedding model, cached reuse is strongly recommended.

In [3]:
vector_store = EmbeddingVectorStore(config)
vector_store.build(chunks)
status('Embedding vector store ready (built or loaded from cache).')

extractor = ConceptExtractor(config.concept_keywords)
graph = KnowledgeNetworkBuilder(extractor).build(documents, chunks)
status(f'Graph built with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.')

[DONE] Embedding vector store ready (built or loaded from cache).
[DONE] Graph built with 30750 nodes and 21492 edges.


## 4. Run baseline vs hybrid retrieval

This step evaluates retrieval quality against HotpotQA supporting titles.

For the current version, relevance is approximated by:
- mapping supporting titles to normalized document IDs
- treating chunks from those supporting documents as relevant chunks


In [4]:
hybrid = HybridRetriever(vector_store, graph)
evaluation_rows = []

for i, sample in enumerate(samples, start=1):
    relevant_doc_ids = {title.lower().replace(' ', '_').replace('/', '_') for title in sample.supporting_titles}
    relevant_chunk_ids = {chunk.chunk_id for chunk in chunks if chunk.doc_id in relevant_doc_ids}

    baseline_results = vector_store.search(sample.question, top_k=config.top_k)
    hybrid_results = hybrid.search(sample.question, top_k=config.top_k, graph_hops=config.graph_hops)

    baseline_metrics = evaluate_retrieval(baseline_results, relevant_chunk_ids)
    hybrid_metrics = evaluate_retrieval(hybrid_results, relevant_chunk_ids)

    evaluation_rows.append({'question_id': sample.question_id, 'category': 'hotpotqa', 'retriever': 'baseline', **baseline_metrics})
    evaluation_rows.append({'question_id': sample.question_id, 'category': 'hotpotqa', 'retriever': 'hybrid', **hybrid_metrics})

    if i <= 3:
        print('=' * 100)
        print(f'Example {i}')
        print('Question:', sample.question)
        print('Supporting titles:', sample.supporting_titles)
        print('Baseline metrics:', baseline_metrics)
        print('Hybrid metrics:  ', hybrid_metrics)

status(f'Completed retrieval evaluation for {len(samples)} HotpotQA questions.')
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df.head())

Example 1
Question: Which Emmett's Mark actor also played in the HBO series "The Wire"?
Supporting titles: ["Emmett's Mark", 'John Doman']
Baseline metrics: {'hit_rate': 1.0, 'recall_at_k': 0.5, 'precision_at_k': 0.3333333333333333, 'mrr': 1.0}
Hybrid metrics:   {'hit_rate': 1.0, 'recall_at_k': 0.5, 'precision_at_k': 0.3333333333333333, 'mrr': 1.0}
Example 2
Question: Who directed the upcoming British action comedy film which has Johnny English as the first part? 
Supporting titles: ['Johnny English (film series)', 'Johnny English 3']
Baseline metrics: {'hit_rate': 1.0, 'recall_at_k': 0.6666666666666666, 'precision_at_k': 0.6666666666666666, 'mrr': 1.0}
Hybrid metrics:   {'hit_rate': 1.0, 'recall_at_k': 0.6666666666666666, 'precision_at_k': 0.6666666666666666, 'mrr': 1.0}
Example 3
Question: What conference has the most valuable NBA franchise according to Forbes?
Supporting titles: ['2008 NBA Playoffs', 'Los Angeles Lakers']
Baseline metrics: {'hit_rate': 1.0, 'recall_at_k': 0.25, 'pre

,question_id,category,retriever,hit_rate,recall_at_k,precision_at_k,mrr
0,5a7bb1315542997c3ec97253,hotpotqa,baseline,1.0,0.500000,0.333333,1.000000
1,5a7bb1315542997c3ec97253,hotpotqa,hybrid,1.0,0.500000,0.333333,1.000000
2,5ae03a3655429942ec259c50,hotpotqa,baseline,1.0,0.666667,0.666667,1.000000
3,5ae03a3655429942ec259c50,hotpotqa,hybrid,1.0,0.666667,0.666667,1.000000
4,5ab77e6655429928e1fe385d,hotpotqa,baseline,1.0,0.250000,0.333333,0.333333


## 5. Save benchmark artifacts

This step saves both query-level and aggregate evaluation outputs so the benchmark can be reviewed outside the notebook.

In [5]:
query_level_path = artifacts.save_dataframe(evaluation_df, f'hotpotqa_query_level_{run_id}.csv')
aggregate_df = evaluation_df.groupby('retriever')[['hit_rate', 'recall_at_k', 'precision_at_k', 'mrr']].mean().reset_index()
aggregate_path = artifacts.save_dataframe(aggregate_df, f'hotpotqa_aggregate_{run_id}.csv')

status(f'Saved HotpotQA query-level results to {query_level_path}')
status(f'Saved HotpotQA aggregate results to {aggregate_path}')
display(aggregate_df)

[DONE] Saved HotpotQA query-level results to C:\Users\minwuu01\GKN\artifacts\reports\hotpotqa_query_level_20260614_161438.csv
[DONE] Saved HotpotQA aggregate results to C:\Users\minwuu01\GKN\artifacts\reports\hotpotqa_aggregate_20260614_161438.csv


,retriever,hit_rate,recall_at_k,precision_at_k,mrr
0,baseline,0.950951,0.572992,0.574241,0.903904
1,hybrid,0.950951,0.574193,0.576243,0.894061


## 6. Final note

Suggested progression:
- `HOTPOTQA_SAMPLE_SIZE=25` for first validation
- `HOTPOTQA_SAMPLE_SIZE=50` for a stronger smoke benchmark
- `HOTPOTQA_SAMPLE_SIZE=100` for a more meaningful public comparison run

Suggested embedding progression:
- `EMBEDDING_CHOICE=local` for fast debugging and zero API quota risk
- `EMBEDDING_CHOICE=small` for cloud benchmark runs
- `EMBEDDING_CHOICE=large` only when higher-quality embeddings are worth the extra cost/quota